# KOIB VK Bot - Google Colab Notebook

Ноутбук для запуска VK-чат-бота по документации КОИБ с использованием RAG и GigaChat.

## Предварительные требования

1. **Настройте Colab Secrets** (в меню слева 🔑 → "Add a new secret"):
   - `GIGACHAT_CREDENTIALS` — учётные данные GigaChat (client_id:client_secret в base64)
   - `VK_GROUP_TOKEN` — токен группы ВКонтакте

2. **Подготовьте документы на Google Drive**:
   - Создайте папку `/content/drive/MyDrive/Koib/docs/`
   - Поместите туда PDF/DOCX файлы документации КОИБ

3. **Создайте system prompt** (опционально):
   - Файл `/content/drive/MyDrive/Koib/prompt.txt`
   - Первый абзац до `---` будет использоваться как приветствие

## Ячейка 1: Установка зависимостей

In [ ]:
# -*- coding: utf-8 -*-
"""
Установка всех необходимых зависимостей для KOIB VK Bot
"""

import sys

# Обновляем pip
!pip install --upgrade pip -q

# Основные зависимости для обработки документов
!pip install pymupdf>=1.23.0 -q
!pip install python-docx>=0.8.11 -q
!pip install Pillow>=9.0.0 -q

# OCR
!pip install pytesseract>=0.3.10 -q
!pip install easyocr>=1.7.0 -q

# ML и эмбеддинги
!pip install numpy>=1.24.0 -q
!pip install sentence-transformers>=2.2.0 -q
!pip install faiss-cpu>=1.7.4 -q

# LangChain
!pip install langchain-core>=0.1.0 -q
!pip install langchain>=0.1.0 -q
!pip install langchain-text-splitters>=0.0.1 -q
!pip install langchain-community>=0.0.10 -q
!pip install langchain-huggingface>=0.0.1 -q

# VK Bot
!pip install vk-api>=11.9.0 -q

# HTTP клиент для GigaChat
!pip install requests>=2.28.0 -q

# Прочее
!pip install tqdm>=4.65.0 -q

print("✅ Все зависимости установлены!")
print(f"Python version: {sys.version}")

## Ячейка 2: Клонирование репозитория и настройка путей

In [ ]:
# -*- coding: utf-8 -*-
"""
Клонирование репозитория и настройка путей
"""

import os
import sys
from pathlib import Path

# Переходим в рабочую директорию
%cd /content

# Клонируем репозиторий (если ещё не склонирован)
if not Path("/content/workspace").exists():
    !git clone https://github.com/ostrakin/Koib.git workspace
else:
    print("✅ Репозиторий уже склонирован")

# Добавляем пути в sys.path
workspace_path = "/content/workspace"
src_path = "/content/workspace/src"

if workspace_path not in sys.path:
    sys.path.insert(0, workspace_path)
if src_path not in sys.path:
    sys.path.insert(0, src_path)

print(f"✅ Пути настроены:")
print(f"   Workspace: {workspace_path}")
print(f"   Src: {src_path}")

# Проверяем наличие модулей
try:
    from setup import initialize
    print("✅ Модуль setup.py найден")
except ImportError as e:
    print(f"⚠️ Ошибка импорта setup: {e}")

try:
    from vk_bot import KoibVKBot
    print("✅ Модуль vk_bot.py найден")
except ImportError as e:
    print(f"⚠️ Ошибка импорта vk_bot: {e}")

## Ячейка 3: Инициализация системы

In [ ]:
# -*- coding: utf-8 -*-
"""
Инициализация RAG системы

При первом запуске:
- Смонтирует Google Drive
- Запросит доступ к секретам
- Построит FAISS-индекс (может занять несколько минут)

При повторных запусках:
- Загрузит существующий индекс (~30-60 секунд)
"""

import logging

# Настраиваем логирование
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)

print("🚀 Начало инициализации системы...")
print("="*60)

# Импортируем функцию инициализации
from setup import initialize

# Запускаем инициализацию
engine, system_prompt = initialize()

print("\n" + "="*60)
print("✅ СИСТЕМА ГОТОВА К РАБОТЕ")
print("="*60)
print(f"\nSystem prompt (первые 200 символов):")
print(system_prompt[:200] + "...\n")

# Сохраняем переменные для следующей ячейки
%store engine
%store system_prompt

print("\nПереходите к следующей ячейке для запуска бота 🚀")

## Ячейка 4: Запуск VK бота

In [ ]:
# -*- coding: utf-8 -*-
"""
Запуск VK бота

Бот будет работать в режиме Long Poll до остановки ячейки.
Для остановки нажмите кнопку "Stop" (квадрат) слева от ячейки.
"""

import logging

# Восстанавливаем переменные из предыдущей ячейки
%store -r engine
%store -r system_prompt

# Импортируем класс бота
from vk_bot import KoibVKBot

# Создаём и запускаем бота
print("🤖 Создание VK бота...")
bot = KoibVKBot(engine, system_prompt)

print("\n" + "="*60)
print("🚀 ЗАПУСК VK БОТА")
print("="*60)
print("\nБот запущен и ожидает сообщения от пользователей...")
print("Для остановки нажмите кнопку 'Stop' (квадрат) слева.")
print("\n" + "-"*60)

# Запускаем Long Poll (блокирующий вызов)
bot.run()

## Полезные команды для отладки

In [ ]:
# -*- coding: utf-8 -*-
"""
Ячейка для тестирования и отладки
"""

# Проверка структуры Google Drive
from pathlib import Path

koib_path = Path("/content/drive/MyDrive/Koib")

print("📁 Структура папки Koib на Google Drive:")
print("="*60)

if koib_path.exists():
    for item in sorted(koib_path.rglob("*")):
        if item.is_file():
            size = item.stat().st_size / 1024  # KB
            print(f"📄 {item.relative_to(koib_path)} ({size:.1f} KB)")
        else:
            print(f"📂 {item.relative_to(koib_path)}/")
else:
    print("❌ Папка Koib не найдена. Запустите ячейку инициализации.")

print("\n" + "="*60)

# Тестовый запрос к движку
print("\n🧪 Тестовый запрос к RAG движку:")
try:
    %store -r engine
    context = engine.ask_with_llm_context("Как включить КОИБ?", koib_model="koib2010")
    print(f"Контекст найден: {len(context)} символов")
    print("\nПервые 500 символов контекста:")
    print(context[:500] + "...")
except Exception as e:
    print(f"❌ Ошибка теста: {e}")